In [105]:
import pandas as pd
from experiment import RollingTestAnalyzer

raw_path = r"rolling_test_raw.csv"
history_df = pd.read_csv(r"data\power_daily_raw.csv")

rolling_df = pd.read_csv(raw_path)
first_target = pd.to_datetime(rolling_df["target_date"]).min()

# 首个 test target_date 之前的真实历史序列
history_actuals = history_df.loc[
    pd.to_datetime(history_df["date"]) < first_target,
    "OT",
].to_numpy(dtype=float)
   
seasonality = 7


a_path = r"timellm\short_term_forecast_power_96_7_gpt2_m22s_TimeLLM_custom_ftMS_tilast_eg1_eh64_sl96_ll48_pl7_dm16_nh8_el2_dl1_df32_fc1_ebtimeF_raw_0-youwaisheng\rolling_test_raw.csv"
b_path = r"timellm\short_term_forecast_power_96_7_gpt2_s_TimeLLM_custom_ftS_tilast_eg0_eh64_sl96_ll48_pl7_dm16_nh8_el2_dl1_df32_fc1_ebtimeF_33101_exog_cfg_0-nowaisheng\rolling_test_raw.csv"
# a_path = r"artifacts\LSTM_both_feat\20260416_154832\rolling_test_raw.csv"
# b_path = r"artifacts\LSTM_no_feat\20260417_130738\rolling_test_raw.csv"

# b_path = r"artifacts\PatchTST_no_feat\20260417_130811\rolling_test_raw.csv"

a = RollingTestAnalyzer(
    rolling_raw=a_path,
    history_actuals=history_actuals,
    seasonality=seasonality,
)

b = RollingTestAnalyzer(
    rolling_raw=b_path,
    history_actuals=history_actuals,
    seasonality=seasonality,
)

# 1. overall 对比
overall_a = a.overall_metrics()
overall_b = b.overall_metrics()

# 2. horizon-wise 对比（手动看两张表）
horizon_a = a.loss_matrix("horizon")
horizon_b = b.loss_matrix("horizon")

# 3. window-wise 对比
window_a = a.loss_matrix("window")
window_b = b.loss_matrix("window")

# A 相对 B 的 window 统计 + win_rate
summary_a_vs_b = a.loss_summary("window", baseline=b)
summary_b = b.loss_summary("window")







In [106]:
print(f'a:\n{overall_a}')
print(f'b:\n{overall_b}')



a:
MASE       1.559641e+00
RMSE       2.618172e+08
WAPE(%)    1.283422e+01
Bias       1.104691e+08
MAPE(%)    1.322197e+01
Name: overall, dtype: float64
b:
MASE       1.578147e+00
RMSE       2.658599e+08
WAPE(%)    1.298650e+01
Bias       1.137933e+08
MAPE(%)    1.338185e+01
Name: overall, dtype: float64


In [107]:
print('horizon_a:')
horizon_a


horizon_a:


,MAE,RMSE,Bias
horizon,,,
1,3.032716e+08,3.730101e+08,2.736916e+08
2,2.306798e+08,2.726013e+08,1.440602e+08
3,2.043118e+08,2.332822e+08,9.316658e+07
4,1.982897e+08,2.234890e+08,7.627186e+07
5,1.818062e+08,2.030034e+08,1.848493e+07
6,1.849045e+08,2.054549e+08,2.066980e+07
7,2.347335e+08,2.803566e+08,1.469387e+08


In [108]:
print('horizon_b:')
horizon_b

horizon_b:


,MAE,RMSE,Bias
horizon,,,
1,3.146603e+08,3.874681e+08,2.901320e+08
2,2.411110e+08,2.875355e+08,1.679748e+08
3,2.089384e+08,2.401995e+08,1.018009e+08
4,1.983961e+08,2.229135e+08,7.282802e+07
5,1.812144e+08,2.021138e+08,1.504636e+07
6,1.858643e+08,2.066592e+08,2.149369e+07
7,2.260612e+08,2.664921e+08,1.272770e+08


In [109]:
print(f'window_a:')
window_a

window_a:


,MAE,RMSE,MAPE(%)
origin_date,,,
2024-10-17,3.869972e+08,3.978104e+08,23.349285
2024-10-18,3.971664e+08,4.099071e+08,24.102818
2024-10-19,3.984142e+08,4.221769e+08,24.412995
2024-10-20,3.885103e+08,4.065488e+08,23.621000
2024-10-21,3.874510e+08,4.001943e+08,23.555746
...,...,...,...
2025-01-01,1.173723e+08,1.278257e+08,6.484611
2025-01-02,1.262083e+08,1.467095e+08,6.866394
2025-01-03,1.405148e+08,1.657149e+08,7.551126


In [110]:
print('window_b')
window_b

window_b


,MAE,RMSE,MAPE(%)
origin_date,,,
2024-10-17,3.945300e+08,4.062817e+08,23.810202
2024-10-18,3.991586e+08,4.135753e+08,24.225405
2024-10-19,4.040496e+08,4.306956e+08,24.766863
2024-10-20,3.909253e+08,4.089407e+08,23.753641
2024-10-21,3.916755e+08,4.044509e+08,23.806779
...,...,...,...
2025-01-01,1.183748e+08,1.294101e+08,6.539213
2025-01-02,1.241182e+08,1.477556e+08,6.745049
2025-01-03,1.418384e+08,1.686700e+08,7.613471


In [111]:
summary_a_vs_b

,mean,median,std,worst10%,max,win_rate
MAE,2.197139e+08,1.788804e+08,1.052932e+08,3.949115e+08,4.007311e+08,0.740741
RMSE,2.395592e+08,2.001177e+08,1.056393e+08,4.102220e+08,4.221769e+08,0.814815
MAPE(%),1.322197e+01,9.944049e+00,6.878231e+00,2.450564e+01,2.529809e+01,0.740741


In [112]:
summary_b

,mean,median,std,worst10%,max
MAE,2.223208e+08,1.793283e+08,1.065153e+08,3.994608e+08,4.084526e+08
RMSE,2.434004e+08,2.022700e+08,1.069475e+08,4.154088e+08,4.306956e+08
MAPE(%),1.338185e+01,9.956174e+00,6.958465e+00,2.478924e+01,2.576161e+01
